# Fixed point

The whole chain in integer arithmetic, as the FPGA computes it, against floating point. First the integer turbo receiver alone (`--fixed`, `include/MSPRS/Fixed.hpp`) on the Gaussian channel, with each word narrowed in turn; then the integer transmitter and front end (`ota/fixed.py`) over the simulated bench (`ota/channel.py`), alone and with the turbo receiver behind them. `L0 = 3`, 8 passes.

Reads the records under `results/fixed` and the floating-point references of `results/quant`, which ran on the same frames. `docs/fixed_point.md` lists every word; the commands are in `docs/build.md`. BER points resting on fewer than 30 error events are dropped.

In [ ]:
import json, pathlib
import numpy as np
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent
FIXED = ROOT / "results" / "fixed"
QUANT = ROOT / "results" / "quant" / "ber"
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "legend.fontsize": 6.5, "xtick.labelsize": 7,
    "ytick.labelsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.3,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": ":",
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

FLOOR = 30       # minimum error events for a BER point to count
TARGET = 1e-4    # where the losses are read
FAMILIES = ("balanced", "unbalanced")

def curve(d):
    """Eb/N0 and BER of one record directory, ordered and filtered."""
    r = sorted((json.loads(f.read_text()) for f in d.glob("snr_*dB.json")), key=lambda x: x["eb_no_db"])
    r = [x for x in r if x["ers_cnt"] >= FLOOR]
    return np.array([x["eb_no_db"] for x in r]), np.array([x["ber"] for x in r])

def crossing(d, target=TARGET):
    """Eb/N0 where the curve crosses `target`, interpolated in log BER."""
    eb, ber = curve(d)
    return float(np.interp(-np.log10(target), -np.log10(ber), eb)) if ber.min() < target < ber.max() else np.nan

def fixed(fam, tag):
    return FIXED / "ber" / f"nsm_L3_{fam}_fixed_{tag}"

def chain(fam, receiver, scenario):
    return FIXED / "chain" / f"nsm_L3_{fam}_{receiver}_{scenario}"

CHOSEN = {f: FIXED / "ber" / f"nsm_L3_{f}_conv_K3_7iters_fixed" for f in FAMILIES}
MAXLOG = {f: QUANT / f"nsm_L3_{f}_conv_K3_7iters_maxlog" for f in FAMILIES}
MAP = {f: QUANT / f"nsm_L3_{f}_conv_K3_7iters" for f in FAMILIES}

## The turbo receiver

Each record narrows words of a wider format, on the same frames as the floating-point max-log-MAP reference. A message step is quoted in nats at Es/N0 = 5 dB; the record's `config.fixed` holds its whole format.

In [ ]:
# (group, record, label): every word wide, then the messages, the extrinsic
# scale and y from that start; then the metrics with 8-bit y and 4-bit messages
# at a step of 1 nat; then that step tuned.
STUDY = [
    ("start", "wide", "every word wide"),
    ("messages", "msg5_step0.5", "5 bits, step 0.5"), ("messages", "msg5_step1", "5 bits, step 1"),
    ("messages", "msg4_step0.5", "4 bits, step 0.5"), ("messages", "msg4_step1", "4 bits, step 1"),
    ("messages", "msg4_step2", "4 bits, step 2"), ("messages", "msg3_step1", "3 bits, step 1"),
    ("messages", "msg3_step2", "3 bits, step 2"), ("messages", "msg4_dec3", "4 bits out, 3 bits back"),
    ("messages", "msg3_dec4", "3 bits out, 4 bits back"),
    ("scale 0.75", "wide_scale_both", "every word wide, both messages"),
    ("scale 0.75", "msg4_scale_eq", "4 bits, equaliser's message"),
    ("scale 0.75", "msg4_scale_dec", "4 bits, decoder's message"),
    ("scale 0.75", "msg4_scale_both", "4 bits, both messages"),
    ("y", "step1.00", "8 bits"), ("y", "y7", "7 bits"), ("y", "y6", "6 bits"),
    *[("branch metric", f"bm{b}", f"{b} bits") for b in (8, 7, 6, 5)],
    *[("state metric", f"sm{b}", f"{b} bits") for b in (10, 9, 8, 7, 6, 5)],
    *[("decoder metric", f"dsm{b}", f"{b} bits") for b in (8, 7, 6, 5)],
    *[("step", f"step{s}", f"{s} nats") for s in ("0.84", "1.00", "1.09", "1.19", "1.30", "1.42")],
    ("chosen", "narrow_step1.19", "narrow metrics, step 1.19"),
]
ref = crossing(MAXLOG["balanced"])
print(f"balanced, float max-log-MAP at 1e-4: {ref:.2f} dB\n")
loss = {}
for group, tag, label in STUDY:
    loss[tag] = crossing(fixed("balanced", tag)) - ref
    print(f"{group:15s} {label:32s} {loss[tag]:+.3f} dB")

In [ ]:
# The chosen format on both families over 3 to 7 dB, against floating point and
# against the same receiver rescaled to the true noise at every Eb/N0 ("sigma").
for fam in FAMILIES:
    c, m, l, s = crossing(CHOSEN[fam]), crossing(MAP[fam]), crossing(MAXLOG[fam]), crossing(fixed(fam, "sigma"))
    print(f"{fam}: at 1e-4 integer {c:.2f} dB, float MAP {m:.2f}, float max-log {l:.2f}: "
          f"loss {c - m:+.3f} / {c - l:+.3f} dB; with the noise known {s - l:+.3f} dB")
    gaps = [crossing(CHOSEN[fam], t) - crossing(fixed(fam, "sigma"), t) for t in (1e-2, 1e-3, 1e-4, 1e-5)]
    print("   constant scale against the noise known, at 1e-2 .. 1e-5: " + " ".join(f"{g:+.3f}" for g in gaps))
    eb, ber = curve(CHOSEN[fam])
    r = [json.loads(f.read_text()) for f in sorted(CHOSEN[fam].glob("snr_*dB.json"))]
    hi = [x for x in r if x["eb_no_db"] >= 6.0 - 1e-9]
    print(f"   6 to 7 dB: {sum(x['ers_cnt'] for x in hi)} errors in {sum(x['bits_cnt'] for x in hi):.2e} bits")

## The front end

The mean squared error of the received symbols against those sent, as a modulation error ratio next to the 2 Es/N0 an ideal receiver reaches on the in-phase rail. Over the bench's received levels with a carrier offset of 20 to 150 Hz and a 1 Hz phase walk, and over phase walks at Es/N0 = 5 dB. The wide twin is the same receiver with every front-end word wide (`Formats.wide`); `sync.py` is the floating-point front end the over-the-air notebook uses.

In [ ]:
FE = json.loads((FIXED / "front_end.json").read_text())
RX = [("fixed", "integer"), ("wide", "wide twin"), ("fixed_no_loop", "integer, no phase loop"), ("float", "sync.py")]
print("loss against the ideal receiver (dB)")
print(" Es/N0 " + "".join(f"{n:>24s}" for _, n in RX) + "   missed")
for r in FE["levels"]:
    print(f"{r['esn0_db']:6.0f} " + "".join(f"{r['ideal_db'] - r[k + '_db']:+24.3f}" for k, _ in RX) + f"   {r['fixed_missed']}")
print("\n linewidth (Hz), Es/N0 = 5 dB")
for r in FE["walks"]:
    print(f"{r['linewidth_hz']:6.0f} " + "".join(f"{r['ideal_db'] - r[k + '_db']:+24.3f}" for k, _ in RX))

## The whole chain

Integer transmitter, simulated bench, integer front end and integer turbo receiver, next to `sync.py` and the MAP turbo receiver on the same captures. "Gaussian" draws a random carrier phase and timing per burst; "bench" adds a carrier offset of 20 to 150 Hz and a 1 Hz phase walk.

In [ ]:
SCENARIOS = ("gaussian", "bench")
for fam in FAMILIES:
    m = crossing(MAP[fam])
    print(f"{fam}: float MAP with ideal sync {m:.2f} dB, integer turbo receiver with ideal sync {crossing(CHOSEN[fam]) - m:+.3f} dB")
    for sc in SCENARIOS:
        row = []
        for rx in ("fixed", "float"):
            d = chain(fam, rx, sc)
            missed = sum(json.loads(f.read_text())["missed"] for f in d.glob("snr_*dB.json"))
            c = crossing(d)
            row.append(f"{rx} " + (f"{c - m:+.3f} dB" if np.isfinite(c) else "does not reach 1e-4") + f" ({missed} missed)")
        print(f"   {sc:8s} " + "   ".join(row))

In [ ]:
FAMILY_COLOR = {"balanced": "#2a78d6", "unbalanced": "#eb6834"}
WIDTH_COLOR = {3: "#86b6ef", 4: "#2a78d6", 5: "#104281"}   # light to dark with the width
WORD_COLOR = {"y": "#2a78d6", "bm": "#eb6834", "sm": "#1baf7a", "dsm": "#eda100"}
WORD_MARKER = {"y": "o", "bm": "s", "sm": "^", "dsm": "D"}
REF = "#000000"    # floating point and other references
OTHER = "#898781"  # sync.py
CHOSEN_WIDTH = {"y": 8, "bm": 7, "sm": 7, "dsm": 5}

fig, axes = plt.subplots(2, 3, figsize=(11.4, 7.0))

ax = axes[0, 0]
steps = {3: [(1, "msg3_step1"), (2, "msg3_step2")],
         4: [(0.5, "msg4_step0.5"), (0.84, "step0.84"), (1.0, "step1.00"), (1.09, "step1.09"), (1.19, "step1.19"),
             (1.30, "step1.30"), (1.42, "step1.42"), (2.0, "msg4_step2")],
         5: [(0.5, "msg5_step0.5"), (1, "msg5_step1")]}
for b, pts in steps.items():
    ax.plot([s for s, _ in pts], [loss[t] for _, t in pts], marker="o", markersize=3.5, color=WIDTH_COLOR[b], label=f"{b} bits")
ref_u = crossing(MAXLOG["unbalanced"])
pts = [(1.0, "step1.00"), (1.19, "step1.19"), (1.42, "step1.42")]
ax.plot([s for s, _ in pts], [crossing(fixed("unbalanced", t)) - ref_u for _, t in pts], marker="s", markersize=3.5,
        ls="--", mfc="white", color=WIDTH_COLOR[4], label="4 bits, unbalanced")
ax.set(xscale="log", xlabel="message step at $E_s/N_0$ = 5 dB (nats)", ylabel="loss at BER $10^{-4}$ (dB)",
       title="Messages, against float max-log-MAP", ylim=(0, 0.9))
ax.set_xticks([0.5, 1, 2], ["0.5", "1", "2"])
ax.xaxis.set_minor_formatter(plt.NullFormatter())
ax.legend(loc="upper right")

ax = axes[0, 1]
words = {"y": [(6, "y6"), (7, "y7"), (8, "step1.00")],
         "bm": [(b, f"bm{b}") for b in (5, 6, 7, 8)],
         "sm": [(b, f"sm{b}") for b in (5, 6, 7, 8, 9, 10)],
         "dsm": [(b, f"dsm{b}") for b in (5, 6, 7, 8)]}
names = {"y": "received symbol", "bm": "branch metric", "sm": "equaliser state metric", "dsm": "decoder state metric"}
for k, pts in words.items():
    ax.plot([b for b, _ in pts], [loss[t] for _, t in pts], marker=WORD_MARKER[k], markersize=4, color=WORD_COLOR[k],
            label=names[k])
    ax.plot([CHOSEN_WIDTH[k]], [loss[dict((b, t) for b, t in pts)[CHOSEN_WIDTH[k]]]], marker="o", markersize=9,
            mfc="none", color=WORD_COLOR[k])
ax.set(xlabel="bits", ylabel="loss at BER $10^{-4}$ (dB)", ylim=(0, 0.9),
       title="Words, with 4-bit messages at step 1 (circled: chosen)")
ax.legend(loc="upper right")

ax = axes[0, 2]
for fam in FAMILIES:
    c = FAMILY_COLOR[fam]
    ax.semilogy(*curve(MAP[fam]), color=c, lw=0.9, ls=":", label=f"{fam}, float MAP")
    ax.semilogy(*curve(fixed(fam, "sigma")), color=c, lw=0.9, ls="--", label=f"{fam}, integer, noise known")
    ax.semilogy(*curve(CHOSEN[fam]), color=c, marker="o", markersize=2.5, label=f"{fam}, integer")
ax.axhline(TARGET, color="0.5", lw=0.6)
ax.set(xlabel="$E_b/N_0$ (dB)", ylabel="BER", title="Integer turbo receiver, Gaussian channel", xlim=(3, 7))
ax.legend(loc="lower left")

ax = axes[1, 0]
fam = "balanced"
ax.semilogy(*curve(MAP[fam]), color=REF, lw=0.9, ls=":", label="float MAP, ideal sync")
ax.semilogy(*curve(CHOSEN[fam]), color=REF, lw=0.9, label="integer turbo, ideal sync")
for sc, ls, mk in (("gaussian", "-", "o"), ("bench", "--", "D")):
    ax.semilogy(*curve(chain(fam, "fixed", sc)), color=FAMILY_COLOR[fam], marker=mk, markersize=3, mfc="white", ls=ls,
                label=f"integer chain, {sc}")
    ax.semilogy(*curve(chain(fam, "float", sc)), color=OTHER, marker=mk, markersize=3, mfc="white", ls=ls,
                label=f"sync.py + MAP, {sc}")
ax.axhline(TARGET, color="0.5", lw=0.6)
ax.set(xlabel="$E_b/N_0$ (dB)", ylabel="BER", title=f"Whole chain, {fam}")
ax.legend(loc="lower left")

ax = axes[1, 1]
es = [r["esn0_db"] for r in FE["levels"]]
STYLE = {"fixed": (FAMILY_COLOR["balanced"], "-", "o"), "wide": (REF, "--", None),
         "fixed_no_loop": (FAMILY_COLOR["unbalanced"], "-", "s"), "float": (OTHER, "-", "D")}
for k, n in RX:
    c, ls, mk = STYLE[k]
    ax.plot(es, [r["ideal_db"] - r[k + "_db"] for r in FE["levels"]], marker=mk, markersize=3.5, color=c, ls=ls,
            lw=0.9 if k == "wide" else 1.3, label=n)
ax.set(xlabel="$E_s/N_0$ (dB)", ylabel="MER loss (dB)", ylim=(0, 2.5),
       title="Front end over the bench's levels, 1 Hz phase walk")
ax.legend(loc="upper left")

ax = axes[1, 2]
lw = [r["linewidth_hz"] for r in FE["walks"]]
x = np.arange(len(lw))
for k, n in RX:
    if k == "wide":
        continue
    c, ls, mk = STYLE[k]
    ax.plot(x, [r["ideal_db"] - r[k + "_db"] for r in FE["walks"]], marker=mk, markersize=3.5, color=c, ls=ls, label=n)
ax.set_xticks(x, [f"{v:g}" for v in lw])
ax.set(xlabel="phase-walk linewidth (Hz)", ylabel="MER loss (dB)", yscale="log",
       title="Front end against the phase walk, $E_s/N_0$ = 5 dB")
ax.legend(loc="upper left")

fig.tight_layout()
fig.savefig(FIGURES / "fixed_point.png")